# 📊 RSI Range Price Behaviour Analysis

**Question:** How does Nifty 50 price behave across different RSI ranges?

**RSI Zones:**
- Zone 1 — RSI < 40  (Oversold territory)
- Zone 2 — RSI 40–60 (Neutral / transitional zone)
- Zone 3 — RSI > 60  (Overbought territory)

**Metrics analysed per zone:**
- Number of days in each zone
- % of UP days vs DOWN days
- Average next-day return
- Average 5-day forward return
- Average 10-day forward return
- Volatility (std of returns)
- Best and worst single-day moves
- Typical price momentum

**Input:** `nifty50_step3_log_return.csv` (raw OHLC + indicators + log return)

---
**Runtime → Run All**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────────────────────────────────────
!pip install pandas numpy scipy -q
print('✓ Libraries ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Upload File and Load Data
# ─────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files

print('Upload: nifty50_step3_log_return.csv')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.sort_values('Date').reset_index(drop=True)

for col in ['Close', 'RSI_14', 'LogReturn_Close']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['RSI_14', 'LogReturn_Close'])

print(f'\n✓ Rows loaded  : {len(df)}')
print(f'✓ Date range   : {df["Date"].iloc[0].date()} → {df["Date"].iloc[-1].date()}')
print(f'✓ RSI range    : {df["RSI_14"].min():.2f} → {df["RSI_14"].max():.2f}')
print(f'✓ Close range  : {df["Close"].min():,.2f} → {df["Close"].max():,.2f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — Assign RSI Zones and Compute Forward Returns
# ─────────────────────────────────────────────────────────────────────────

# ── Assign RSI zone to each day ────────────────────────────────────────────
def assign_zone(rsi):
    if rsi < 40:
        return 'Zone 1: RSI < 40 (Oversold)'
    elif rsi <= 60:
        return 'Zone 2: RSI 40–60 (Neutral)'
    else:
        return 'Zone 3: RSI > 60 (Overbought)'

df['RSI_Zone']  = df['RSI_14'].apply(assign_zone)

# ── Forward returns ────────────────────────────────────────────────────────
# Next-day log return (how does price move the DAY AFTER this RSI reading)
df['next_1d_return']  = df['LogReturn_Close'].shift(-1)

# 5-day cumulative forward return
df['next_5d_return']  = sum(
    df['LogReturn_Close'].shift(-i) for i in range(1, 6)
)

# 10-day cumulative forward return
df['next_10d_return'] = sum(
    df['LogReturn_Close'].shift(-i) for i in range(1, 11)
)

# Direction of next-day move
df['next_direction']  = df['next_1d_return'].apply(
    lambda x: 'UP' if x > 0 else ('DOWN' if x < 0 else 'FLAT')
)

# Drop last rows where forward returns are NaN
df_clean = df.dropna(subset=['next_1d_return','next_5d_return','next_10d_return'])

print(f'  RSI zone distribution:')
zone_counts = df_clean['RSI_Zone'].value_counts().sort_index()
for zone, count in zone_counts.items():
    pct = count / len(df_clean) * 100
    bar = '█' * int(pct / 2)
    print(f'    {zone}: {count:>4} days ({pct:.1f}%)  {bar}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Full Analysis Per RSI Zone
# ─────────────────────────────────────────────────────────────────────────

ZONES = [
    ('Zone 1: RSI < 40 (Oversold)',    'RSI < 40',  '#e74c3c'),
    ('Zone 2: RSI 40–60 (Neutral)',    'RSI 40–60', '#f39c12'),
    ('Zone 3: RSI > 60 (Overbought)', 'RSI > 60',  '#27ae60'),
]

results = []

print('=' * 75)
print('  NIFTY 50 PRICE BEHAVIOUR BY RSI ZONE')
print(f'  Period: {df_clean["Date"].iloc[0].date()} → {df_clean["Date"].iloc[-1].date()}')
print('=' * 75)

for zone_label, zone_short, _ in ZONES:
    zone_df = df_clean[df_clean['RSI_Zone'] == zone_label].copy()

    if len(zone_df) == 0:
        continue

    n          = len(zone_df)
    up_days    = (zone_df['next_direction'] == 'UP').sum()
    down_days  = (zone_df['next_direction'] == 'DOWN').sum()
    up_pct     = up_days / n * 100
    down_pct   = down_days / n * 100

    avg_1d     = zone_df['next_1d_return'].mean()
    avg_5d     = zone_df['next_5d_return'].mean()
    avg_10d    = zone_df['next_10d_return'].mean()
    std_1d     = zone_df['next_1d_return'].std()
    best_1d    = zone_df['next_1d_return'].max()
    worst_1d   = zone_df['next_1d_return'].min()

    avg_rsi    = zone_df['RSI_14'].mean()
    avg_close  = zone_df['Close'].mean()

    # T-test: is avg next-day return significantly different from 0?
    t_stat, p_val = stats.ttest_1samp(zone_df['next_1d_return'].dropna(), 0)

    results.append({
        'zone':       zone_short,
        'n':          n,
        'up_pct':     up_pct,
        'down_pct':   down_pct,
        'avg_1d':     avg_1d,
        'avg_5d':     avg_5d,
        'avg_10d':    avg_10d,
        'std_1d':     std_1d,
        'best_1d':    best_1d,
        'worst_1d':   worst_1d,
        'avg_rsi':    avg_rsi,
        'p_val':      p_val,
    })

    print(f'\n  ┌─ {zone_label} ─{'─' * max(0, 45 - len(zone_label))}┐')
    print(f'  │  Days in zone      : {n:>4} days  ({n/len(df_clean)*100:.1f}% of all trading days)')
    print(f'  │  Average RSI       : {avg_rsi:.1f}')
    print(f'  │  Average Close     : {avg_close:,.2f}')
    print(f'  │')
    print(f'  │  NEXT-DAY DIRECTION:')
    print(f'  │    UP   days : {up_days:>4} ({up_pct:.1f}%)')
    print(f'  │    DOWN days : {down_days:>4} ({down_pct:.1f}%)')
    print(f'  │')
    print(f'  │  FORWARD RETURNS (log return, multiply by 100 for %):')
    print(f'  │    Next 1-day   avg : {avg_1d:+.4f}  ({avg_1d*100:+.2f}%)')
    print(f'  │    Next 5-day   avg : {avg_5d:+.4f}  ({avg_5d*100:+.2f}%)')
    print(f'  │    Next 10-day  avg : {avg_10d:+.4f}  ({avg_10d*100:+.2f}%)')
    print(f'  │')
    print(f'  │  VOLATILITY & EXTREMES:')
    print(f'  │    Daily std dev    : {std_1d:.4f}  ({std_1d*100:.2f}%)')
    print(f'  │    Best  next-day   : {best_1d:+.4f}  ({best_1d*100:+.2f}%)')
    print(f'  │    Worst next-day   : {worst_1d:+.4f}  ({worst_1d*100:+.2f}%)')
    print(f'  │')
    sig = '✅ Significant' if p_val < 0.05 else '— Not significant'
    print(f'  │  STATISTICAL TEST:')
    print(f'  │    t-test (return ≠ 0): p = {p_val:.4f}  {sig}')
    print(f'  └{'─' * 55}┘')

print()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — Summary Comparison Table
# ─────────────────────────────────────────────────────────────────────────

print('=' * 75)
print('  SUMMARY COMPARISON TABLE')
print('=' * 75)
print(f'  {"RSI Zone":<20} {"Days":>5} {"UP%":>7} {"1d Ret%":>9} {"5d Ret%":>9} {"10d Ret%":>10} {"Volatility":>11}')
print('  ' + '─' * 73)

for r in results:
    print(f'  {r["zone"]:<20} '
          f'{r["n"]:>5} '
          f'{r["up_pct"]:>6.1f}% '
          f'{r["avg_1d"]*100:>+8.2f}% '
          f'{r["avg_5d"]*100:>+8.2f}% '
          f'{r["avg_10d"]*100:>+9.2f}% '
          f'{r["std_1d"]*100:>10.2f}%')

print()
print('  KEY INSIGHTS:')
print()

# Find best and worst zones
best_zone  = max(results, key=lambda x: x['avg_1d'])
worst_zone = min(results, key=lambda x: x['avg_1d'])
most_up    = max(results, key=lambda x: x['up_pct'])

print(f'  1. Best avg next-day return  : {best_zone["zone"]}  ({best_zone["avg_1d"]*100:+.2f}%/day)')
print(f'  2. Worst avg next-day return : {worst_zone["zone"]}  ({worst_zone["avg_1d"]*100:+.2f}%/day)')
print(f'  3. Highest UP day frequency  : {most_up["zone"]}  ({most_up["up_pct"]:.1f}% of days go UP next)')
print()

# Interpretation
r1 = results[0] if len(results) > 0 else None
r3 = results[2] if len(results) > 2 else None

if r1 and r3:
    if r1['up_pct'] > r3['up_pct']:
        print('  → CONTRARIAN SIGNAL: RSI < 40 (oversold) leads to more UP days than RSI > 60')
        print('    This supports mean reversion — oversold stocks tend to bounce back')
    else:
        print('  → MOMENTUM SIGNAL: RSI > 60 (overbought) leads to more UP days than RSI < 40')
        print('    This supports momentum — strong stocks continue rising')
    print()
    if abs(r1['std_1d']) > abs(r3['std_1d']):
        print('  → VOLATILITY: RSI < 40 zone is MORE volatile than RSI > 60 zone')
        print('    Oversold conditions create larger price swings in both directions')
    else:
        print('  → VOLATILITY: RSI > 60 zone is MORE volatile than RSI < 40 zone')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Deep Dive: RSI Crossing Zones
#
# What happens when RSI CROSSES from one zone to another?
# This is when the strongest signals occur
# ─────────────────────────────────────────────────────────────────────────

print('=' * 75)
print('  RSI ZONE CROSSING ANALYSIS')
print('  (What happens when RSI crosses into a zone — strongest signals)')
print('=' * 75)

# Detect zone transitions
df_clean = df_clean.copy()
df_clean['zone_num'] = df_clean['RSI_Zone'].map({
    'Zone 1: RSI < 40 (Oversold)':   1,
    'Zone 2: RSI 40–60 (Neutral)':    2,
    'Zone 3: RSI > 60 (Overbought)': 3,
})
df_clean['prev_zone'] = df_clean['zone_num'].shift(1)
df_clean['zone_cross'] = df_clean['zone_num'] != df_clean['prev_zone']

# Days when RSI first enters oversold (<40) from neutral
enter_oversold = df_clean[
    (df_clean['zone_num'] == 1) & (df_clean['prev_zone'] == 2)
]
# Days when RSI exits oversold (crosses back to neutral from <40)
exit_oversold  = df_clean[
    (df_clean['zone_num'] == 2) & (df_clean['prev_zone'] == 1)
]
# Days when RSI enters overbought (>60) from neutral
enter_overbought = df_clean[
    (df_clean['zone_num'] == 3) & (df_clean['prev_zone'] == 2)
]
# Days when RSI exits overbought (crosses back to neutral from >60)
exit_overbought  = df_clean[
    (df_clean['zone_num'] == 2) & (df_clean['prev_zone'] == 3)
]

crossings = [
    ('RSI crosses INTO oversold (<40)',    enter_oversold,   'Bearish signal'),
    ('RSI exits oversold (back to 40+)',   exit_oversold,    'Bullish reversal signal'),
    ('RSI crosses INTO overbought (>60)',  enter_overbought, 'Bullish signal'),
    ('RSI exits overbought (back to <60)', exit_overbought,  'Bearish reversal signal'),
]

for label, cross_df, signal_type in crossings:
    n = len(cross_df)
    if n == 0:
        print(f'\n  {label}: 0 occurrences')
        continue
    up_pct  = (cross_df['next_direction'] == 'UP').mean() * 100
    avg_ret = cross_df['next_1d_return'].mean() * 100
    avg_5d  = cross_df['next_5d_return'].mean() * 100
    print(f'\n  {label}')
    print(f'    Signal type        : {signal_type}')
    print(f'    Occurrences        : {n} times')
    print(f'    Next day UP        : {up_pct:.1f}%')
    print(f'    Avg next-day return: {avg_ret:+.2f}%')
    print(f'    Avg 5-day  return  : {avg_5d:+.2f}%')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — Save Results and Download
# ─────────────────────────────────────────────────────────────────────────
import os

# ── Save summary CSV ───────────────────────────────────────────────────────
summary_rows = []
for zone_label, zone_short, _ in ZONES:
    zone_df = df_clean[df_clean['RSI_Zone'] == zone_label]
    if len(zone_df) == 0:
        continue
    summary_rows.append({
        'RSI_Zone':            zone_short,
        'Days':                len(zone_df),
        'Pct_of_total':        round(len(zone_df)/len(df_clean)*100, 1),
        'UP_pct':              round((zone_df['next_direction']=='UP').mean()*100, 1),
        'DOWN_pct':            round((zone_df['next_direction']=='DOWN').mean()*100, 1),
        'Avg_next_1d_return':  round(zone_df['next_1d_return'].mean()*100, 4),
        'Avg_next_5d_return':  round(zone_df['next_5d_return'].mean()*100, 4),
        'Avg_next_10d_return': round(zone_df['next_10d_return'].mean()*100, 4),
        'Volatility_1d':       round(zone_df['next_1d_return'].std()*100, 4),
        'Best_next_day':       round(zone_df['next_1d_return'].max()*100, 4),
        'Worst_next_day':      round(zone_df['next_1d_return'].min()*100, 4),
        'Avg_RSI':             round(zone_df['RSI_14'].mean(), 2),
        'Avg_Close':           round(zone_df['Close'].mean(), 2),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('rsi_zone_analysis.csv', index=False)

# ── Save full day-level data with zones ────────────────────────────────────
detail_cols = ['Date','Close','RSI_14','RSI_Zone',
               'LogReturn_Close','next_1d_return',
               'next_5d_return','next_10d_return','next_direction']
detail_df = df_clean[detail_cols].copy()
detail_df['Date'] = detail_df['Date'].dt.strftime('%d-%b-%Y')
for col in ['next_1d_return','next_5d_return','next_10d_return','LogReturn_Close']:
    detail_df[col] = (detail_df[col] * 100).round(4)
detail_df = detail_df.rename(columns={
    'next_1d_return':  'next_1d_return_pct',
    'next_5d_return':  'next_5d_return_pct',
    'next_10d_return': 'next_10d_return_pct',
    'LogReturn_Close': 'log_return_pct',
})
detail_df.to_csv('rsi_zone_daily_detail.csv', index=False)

print('=' * 60)
print('  FILES SAVED')
print('=' * 60)
for fname in ['rsi_zone_analysis.csv','rsi_zone_daily_detail.csv']:
    size = os.path.getsize(fname)
    print(f'  ✓ {fname}  ({size/1024:.1f} KB)')

print()
print('  Summary table:')
print(summary_df.to_string(index=False))

print()
print('⬇️  Downloading...')
files.download('rsi_zone_analysis.csv')
files.download('rsi_zone_daily_detail.csv')
print('✓ Both files downloaded')
print()
print('✅ RSI Zone Analysis Complete!')